# Regional MG evidence visualization

This notebook turns overlapping local-evidence windows into calibrated, multi-scale token evidence. It deliberately keeps the score interpretation explicit: the token values are projected window evidence, not token-level classifier probabilities.

Expected scored-window columns are `start_token`, `stop_token`, `window_length`, and `mg_score`. A held-out calibration frame should additionally contain `label`, where HG is `0` and MG is `1`.

In [ ]:
import os
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import Normalize
from matplotlib.patches import Rectangle
from IPython.display import display
from notebooks.utils.local_evidence_models_utils import (
    LOCAL_FEATURE_COLUMNS,
    fit_local_evidence_classifier,
    score_local_evidence_classifier,
)
from notebooks.utils.local_evidence_utils import split_local_evidence_texts
from notebooks.utils.token_candidate_similarity_heatmaps_utils import load_token_labels

from notebooks.utils.regional_mg_evidence_utils import (
    build_multiscale_token_evidence,
    extract_hysteresis_regions,
    fit_probability_calibrator,
    summarize_window_contributions,
)
from notebooks.utils.semeval_boundary_utils import gold_token_boundary

DATA_ROOT = Path(os.environ.get('TEXT_DETECTION_DATA', 'D:/Projects/master_diploma/text-detection-data'))
PROCESSED_ROOT = DATA_ROOT / 'processed'
DATASET = 'semeval2024_task8_subtask_c_train'
SCORER_MODEL = 'gemma_3_1b_it'
TEXT_SPLIT = 'MG'
TEXT_IDS = [0, 1, 2]
ACTIVE_TEXT_ID = TEXT_IDS[0]
SOURCE_DATASETS = ('kaggle_train', 'mage_train', 'outfox_chatgpt_train')
SEMEVAL_SEED = 42
SEMEVAL_ANNOTATIONS_PATH = (
    DATA_ROOT / 'raw/semevals_2024-task8/SemEval2024-task8/subtaskC/'
    / 'SemEval2024_task8_C_train_MG.csv'
)
EVALUATION_PARTITION = 'test'
CLASSIFICATION_RUN_ID = 'feature-vector-classification-2026-09-25-logreg-random-forest-max600-min100'
CLASSIFIER = 'logistic_regression'
FEATURE_SET = 'extended'
TRAINING_REGIME = 'Kaggle'
WINDOW_LENGTHS = (32,)

PREDICTION_ROOT = DATA_ROOT / 'experiment-results' / 'classification' / 'evaluations' / CLASSIFICATION_RUN_ID / 'predictions'
def prediction_path(window_length, partition):
    model_dir = (
        f'{CLASSIFIER}__{SCORER_MODEL}__{FEATURE_SET}__{TRAINING_REGIME}__'
        f'{window_length}__{CLASSIFICATION_RUN_ID}'
    )
    return PREDICTION_ROOT / model_dir / TRAINING_REGIME / str(window_length) / DATASET / partition / 'data.parquet'

SCORED_WINDOWS_PATH = tuple(prediction_path(length, EVALUATION_PARTITION) for length in WINDOW_LENGTHS)
CALIBRATION_WINDOWS_PATH = tuple(prediction_path(length, 'validation') for length in WINDOW_LENGTHS)
TOKENS_PATH = 'stage1_lookup'  # Replace with a token-frame path when needed.

WINDOW_THRESHOLD = 0.50
REGION_START_THRESHOLD = 0.75
REGION_CONTINUE_THRESHOLD = 0.55
REGION_MINIMUM_LENGTH = 3
REGION_MERGE_GAP = 1


In [ ]:
def _load_optional_frame(path, existing_name):
    if path is not None:
        paths = path if isinstance(path, (tuple, list)) else (path,)
        frames = []
        for index, item in enumerate(paths):
            item = Path(item)
            frame = pd.read_parquet(item) if item.suffix == '.parquet' else pd.read_pickle(item)
            if 'window_length' not in frame:
                frame['window_length'] = WINDOW_LENGTHS[index] if len(paths) == len(WINDOW_LENGTHS) else frame['stop_token'] - frame['start_token']
            frames.append(frame)
        return pd.concat(frames, ignore_index=True)
    if existing_name in globals():
        return globals()[existing_name].copy()
    raise ValueError(f'Provide {existing_name} or set its input path')

if DATASET.startswith('semeval'):
    annotations = pd.read_csv(SEMEVAL_ANNOTATIONS_PATH)
    selected_annotations = annotations.loc[annotations['id'].isin(TEXT_IDS)].copy()
    missing_annotations = sorted(set(TEXT_IDS) - set(selected_annotations['id']))
    if missing_annotations:
        raise ValueError(f'Missing SemEval annotations for TEXT_IDS={missing_annotations}')
    token_labels = load_token_labels(PROCESSED_ROOT, DATASET, SCORER_MODEL, TEXT_SPLIT)
    missing_tokens = sorted(set(TEXT_IDS) - set(token_labels))
    if missing_tokens:
        raise ValueError(f'Missing processed token labels for TEXT_IDS={missing_tokens}')
    semeval_boundaries = selected_annotations[['id', 'text', 'boundary_word_index']].copy()
    semeval_boundaries['text_id'] = semeval_boundaries['id'].astype(int)
    semeval_boundaries['boundary_token_position'] = [
        gold_token_boundary(row.text, row.boundary_word_index, token_labels[int(row.id)])
        for row in semeval_boundaries.itertuples(index=False)
    ]
    semeval_boundaries['token_count'] = semeval_boundaries['text_id'].map(
        lambda text_id: len(token_labels[int(text_id)])
    )
    source_windows = pd.concat(
        [
            pd.read_parquet(
                PROCESSED_ROOT / f'{dataset}__{SCORER_MODEL}' / 'local_evidence' / 'windows.parquet'
            )
            for dataset in SOURCE_DATASETS
        ],
        ignore_index=True,
    )
    partitions = split_local_evidence_texts(source_windows, seed=SEMEVAL_SEED)
    train_windows = source_windows.merge(
        partitions['train'][['split', 'text_id', 'label']],
        on=['split', 'text_id', 'label'], validate='many_to_one',
    )
    semeval_windows = pd.read_parquet(
        PROCESSED_ROOT / f'{DATASET}__{SCORER_MODEL}' / 'local_evidence' / 'windows.parquet'
    )
    window_scores = pd.concat(
        [
            score_local_evidence_classifier(
                fit_local_evidence_classifier(
                    CLASSIFIER, train_windows.loc[train_windows['window_length'].eq(length)],
                    LOCAL_FEATURE_COLUMNS, SEMEVAL_SEED,
                ),
                semeval_windows.loc[
                    semeval_windows['window_length'].eq(length)
                    & semeval_windows['text_id'].isin(TEXT_IDS)
                ],
                LOCAL_FEATURE_COLUMNS,
            )
            for length in WINDOW_LENGTHS
        ],
        ignore_index=True,
    )
    calibration_status = 'raw classifier probabilities'
else:
    window_scores = _load_optional_frame(SCORED_WINDOWS_PATH, 'window_scores')
required = {'start_token', 'stop_token', 'window_length', 'mg_score'}
missing = required - set(window_scores.columns)
if missing:
    raise ValueError(f'scored windows are missing columns: {sorted(missing)}')

if 'text_id' in window_scores.columns and DATASET.startswith('semeval'):
    window_scores = window_scores.loc[window_scores['text_id'].isin(TEXT_IDS)].copy()
    if window_scores.empty:
        raise ValueError(f'No scored windows found for TEXT_IDS={TEXT_IDS}')

if TOKENS_PATH != 'stage1_lookup':
    token_frame = _load_optional_frame(TOKENS_PATH, 'tokens')
    tokens = token_frame['token'].astype(str).tolist() if isinstance(token_frame, pd.DataFrame) else list(token_frame)
elif 'tokens' in globals():
    tokens = list(tokens)
else:
    token_labels = load_token_labels(PROCESSED_ROOT, DATASET, SCORER_MODEL, TEXT_SPLIT)
    if ACTIVE_TEXT_ID not in token_labels:
        raise KeyError(f'No {TEXT_SPLIT} token row found for TEXT_ID={ACTIVE_TEXT_ID}')
    tokens = token_labels[ACTIVE_TEXT_ID]

token_count = len(tokens)
window_scores = window_scores.loc[window_scores['start_token'].lt(token_count)].copy()
window_scores['stop_token'] = window_scores['stop_token'].clip(upper=token_count)
display(window_scores.head())
print(f'{len(window_scores):,} windows over {token_count:,} tokens')

## Calibrate held-out window scores

Calibration must use held-out windows, ideally from validation parent texts. If no calibration frame is supplied, the notebook preserves the raw score and labels the result as uncalibrated.

In [ ]:
if DATASET.startswith('semeval'):
    print(calibration_status)
elif CALIBRATION_WINDOWS_PATH is not None or 'calibration_windows' in globals():
    calibration_windows = _load_optional_frame(CALIBRATION_WINDOWS_PATH, 'calibration_windows')
    if 'label' not in calibration_windows:
        raise ValueError('calibration windows must contain label')
    calibrator = fit_probability_calibrator(
        calibration_windows['mg_score'].to_numpy(),
        calibration_windows['label'].to_numpy(),
    )
    window_scores['mg_score'] = calibrator.predict(window_scores['mg_score'].to_numpy())
    calibration_status = 'calibrated on held-out windows'
else:
    warnings.warn('No held-out calibration frame supplied; using raw mg_score values.')
    calibration_status = 'raw, not calibrated'
print(calibration_status)

In [ ]:
if DATASET.startswith('semeval'):
    figure, axes = plt.subplots(
        len(semeval_boundaries), 1, figsize=(16, max(3, 3.2 * len(semeval_boundaries))),
        sharex=False, squeeze=False, constrained_layout=True,
    )
    for axis, boundary in zip(axes.flat, semeval_boundaries.itertuples(index=False), strict=True):
        text_id = int(boundary.text_id)
        text_windows = window_scores.loc[window_scores['text_id'].eq(text_id)]
        text_evidence = build_multiscale_token_evidence(
            text_windows, token_count=int(boundary.token_count),
            window_threshold=WINDOW_THRESHOLD,
        )
        axis.plot(text_evidence['token_position'], text_evidence['robust_score'], color='black')
        axis.axvspan(0, boundary.boundary_token_position, color='steelblue', alpha=0.08, label='HG')
        axis.axvspan(boundary.boundary_token_position, boundary.token_count, color='darkorange', alpha=0.08, label='MG')
        axis.axvline(boundary.boundary_token_position, color='darkorange', linestyle='--', linewidth=1.5, label='gold boundary')
        axis.set_xlim(0, boundary.token_count)
        axis.set_ylim(-0.02, 1.02)
        axis.set_ylabel('MG evidence')
        axis.set_title(f'SemEval MG text {text_id}: gold HG/MG boundary at token {boundary.boundary_token_position}')
    axes[-1, 0].set_xlabel('token position')
    handles, labels = axes[0, 0].get_legend_handles_labels()
    figure.legend(handles, labels, loc='upper right')
    figure.suptitle('SemEval gold HG/MG boundaries, 32-token windows')
    plt.show()

single_window_scores = window_scores.loc[
    window_scores['text_id'].eq(ACTIVE_TEXT_ID)
] if DATASET.startswith('semeval') else window_scores
evidence = build_multiscale_token_evidence(
    single_window_scores,
    token_count=token_count,
    window_threshold=WINDOW_THRESHOLD,
)

regions = extract_hysteresis_regions(
    evidence,
    score_column='robust_score',
    start_threshold=REGION_START_THRESHOLD,
    continue_threshold=REGION_CONTINUE_THRESHOLD,
    minimum_length=REGION_MINIMUM_LENGTH,
    merge_gap=REGION_MERGE_GAP,
)

display(evidence.sort_values('robust_score', ascending=False).head(20))
display(regions)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(16, 9), sharex=True, constrained_layout=True)
x = evidence['token_position'].to_numpy()
axes[0].plot(x, evidence['mean_score'], label='mean window evidence')
axes[0].plot(x, evidence['max_score'], alpha=0.7, label='max window evidence')
axes[0].plot(x, evidence['robust_score'], linewidth=2, label='robust multi-scale evidence')
axes[0].axhline(REGION_START_THRESHOLD, color='black', linestyle='--', linewidth=0.8)
axes[0].set_ylabel('MG evidence')
axes[0].legend(loc='upper right')
axes[1].plot(x, evidence['length_agreement'], color='tab:purple', label='window-length agreement')
axes[1].plot(x, evidence['evidence_fraction'], color='tab:orange', label='windows above threshold')
axes[1].set_ylim(-0.02, 1.02)
axes[1].set_ylabel('agreement')
axes[1].legend(loc='upper right')
axes[2].plot(x, evidence['support_count'], color='tab:green', label='support count')
axes[2].plot(x, evidence['score_std'], color='tab:red', label='score disagreement')
axes[2].set_xlabel('token position')
axes[2].set_ylabel('coverage / spread')
axes[2].legend(loc='upper right')
for region in regions.itertuples(index=False):
    for axis in axes:
        axis.axvspan(region.start_token, region.stop_token, color='red', alpha=0.08)
fig.suptitle(f'Regional MG evidence ({calibration_status})')
plt.show()

In [ ]:
# Text-level view: dark color means stronger robust evidence; gray means no support.
fig, axis = plt.subplots(figsize=(18, max(3, 0.32 * (token_count // 40 + 1))))
normalizer = Normalize(vmin=0, vmax=1, clip=True)
cmap = plt.get_cmap('Reds')
columns = 40
for position, token in enumerate(tokens):
    row, column = divmod(position, columns)
    score = evidence.loc[position, 'robust_score']
    face = cmap(normalizer(score)) if pd.notna(score) else (0.85, 0.85, 0.85, 1)
    axis.add_patch(Rectangle((column, -row - 0.82), 1, 0.72, facecolor=face, edgecolor='white', linewidth=0.3))
    axis.text(column + 0.03, -row - 0.5, str(token).replace('\n', ' '), fontsize=7, va='center', clip_on=True)
for region in regions.itertuples(index=False):
    start_row, start_column = divmod(region.start_token, columns)
    stop_row, stop_column = divmod(region.stop_token - 1, columns)
    if start_row == stop_row:
        axis.add_patch(Rectangle((start_column, -start_row - 0.88), region.token_count, 0.84, fill=False, edgecolor='black', linewidth=1.5))
axis.set_xlim(0, columns)
axis.set_ylim(-((token_count - 1) // columns + 1), 0)
axis.axis('off')
axis.set_title('Token evidence and detected regions')
plt.show()

In [ ]:
# Explain the strongest detected region using the model's signed window contributions.
contribution_columns = [column for column in single_window_scores if column.startswith('contribution__')]
if regions.empty:
    print('No regions passed the current hysteresis thresholds.')
elif not contribution_columns:
    print('No contribution columns are present in the scored-window frame.')
else:
    target = regions.iloc[0]
    overlapping = single_window_scores.loc[
        single_window_scores['start_token'].lt(target['stop_token'])
        & single_window_scores['stop_token'].gt(target['start_token'])
    ]
    display(summarize_window_contributions(overlapping, top_n=10))

### Interpretation

- `mean_score` is stable overlapping-window evidence.
- `max_score` highlights localized suspicious evidence.
- `length_agreement` reports the fraction of available window lengths supporting MG evidence.
- `robust_score` is `mean_score * length_agreement`; it suppresses evidence seen at only one scale.
- `score_std` and low support identify regions that should not be over-interpreted.
- Region boundaries are generated with separate start and continuation thresholds, so short dips do not fragment a region.